### `parse.py`

In [16]:
import math
import operator as op

In [17]:
# Type Definitions
SYMBOL = str
NUMBER = (int, float)     # int or float
LIST   = list
ATOM   = (SYMBOL, NUMBER) # symbol or number
EXP    = (ATOM, LIST)     # Expression will either be atom or list
ENV    = Environment()    # Environment will be represented as an object 

In [18]:
def tokenize(string: str) -> list:
    """
    Convert a string into a list of tokens.
    
    Parameters:
    string (str): target string to turn into a list of tokens separated by space.

    Returns:
    list: the list of tokens.
    """
    # Add spaces to paranthesis so that it is a separate token
    return string.replace('(',' ( ').replace(')', ' ) ').split()     

In [25]:
def structure_tokens(tokens: list) -> EXP:
    """
    Convert list of tokens into lisp expressions recursively

    Parameters:
    tokens (list): list of tokens to convert

    Returns:
    EXP: lisp expression
    """
    if len(tokens) == 0:
        raiseSyntaxError('unexpected EOF')
    token = tokens.pop(0)
    if token == '(':
        L = []
        while tokens[0] != ')':
            L.append(structure_tokens(tokens))
        tokens.pop(0) # tokens[0] is ) so pop it
        return L 
    elif token == ')':
        raise SyntaxError('unexpected )')
    else:
        return to_atom(token)

In [26]:
def to_atom(token: str) -> ATOM:
    """
    Convert token to a lisp atom

    Parameters:
    token (str): target token to convert

    Returns:
    ATOM: lisp atom
    """
    try: 
        return int(token)
    except ValueError:
        try:
            return float(token)
        except ValueError:
            return SYMBOL(token)
    

In [36]:
def parse(raw_expression: str) -> EXP:
    """
    Convert a string into a lisp expression
    
    Parameters:
    raw_expression (str): raw expression as a string to convert into a lisp expression

    Returns:
    EXP: lisp expression
    """
    return structure_tokens(tokenize(raw_expression))

### `Environment.py`

In [161]:
class Environment: 
    def __init__(self, outer=None):
        self.variables = {}
        self.outer = outer
        
    def find(self, var):
        if var in self.variables:
            return self
        elif self.outer:
            return self.outer.find(var)
        else:
            raise NameError(f"Undefined variable: {var}")

### `Procedure.py`

In [162]:
class Procedure(object):
    def __init__(self, params, body, env):
        self.params = params
        self.body = body
        self.env = env
    def __call__(self, *args):
        inner_env = Environment(outer=self.env)
        for param, arg in zip(self.params, args):
            inner_env.variables[param] = arg
        return eval(self.body, inner_env)

### `evaluate.py`

### TEST

In [163]:
global_env = Environment()
global_env.variables = {
        '+':op.add, '-':op.sub, '*':op.mul, '/':op.truediv, 
        '>':op.gt, '<':op.lt, '>=':op.ge, '<=':op.le, '=':op.eq, 
        'abs':     abs,
        'append':  op.add,  
        'apply':   lambda proc, args: proc(*args),
        'begin':   lambda *x: x[-1],
        'car':     lambda x: x[0],
        'cdr':     lambda x: x[1:], 
        'cons':    lambda x,y: [x] + y,
        'eq?':     op.is_, 
        'expt':    pow,
        'equal?':  op.eq, 
        'length':  len, 
        'list':    lambda *x: LIST(x), 
        'list?':   lambda x: isinstance(x, LIST), 
        'map':     map,
        'max':     max,
        'min':     min,
        'not':     op.not_,
        'null?':   lambda x: x == [], 
        'number?': lambda x: isinstance(x, NUMBER),  
		'print':   print,
        'procedure?': callable,
        'round':   round,
        'symbol?': lambda x: isinstance(x, SYMBOL),
}
global_env.variables.update(vars(math)) # sin, ...

def eval(x: EXP, env=global_env) -> EXP:
    if isinstance(x, SYMBOL):    # variable reference
        return env.find(x).variables[x]
    elif not isinstance(x, LIST):# constant 
        return x   
    op, *args = x       
    if op == 'quote':            # quotation
        return args[0]
    elif op == 'if':             # conditional
        (test, conseq, alt) = args
        exp = (conseq if eval(test, env) else alt)
        return eval(exp, env)
    elif op == 'define':         # definition
        (symbol, exp) = args
        env.variables[symbol] = eval(exp, env)
    elif op == 'set!':           # assignment
        (symbol, exp) = args
        env.find(symbol).variables[symbol] = eval(exp, env)
    elif op == 'lambda':         # procedure
        (params, body) = args
        return Procedure(params, body, env)
    else:                        # procedure call
        proc = eval(op, env)
        vals = [eval(arg, env) for arg in args]
        return proc(*vals) 

In [164]:
eval(parse("(define r 10)"))

In [165]:
eval(parse("(* pi (* r r))"))

314.1592653589793

In [166]:
eval(parse("(define circle-area (lambda (r) (* pi (* r r))))"))

In [167]:
eval(parse("(circle-area 3)"))

28.274333882308138

In [171]:
eval(parse("(define first car)"))

In [173]:
eval(parse(" (define rest cdr)"))

In [174]:
eval(parse("(define count (lambda (item L) (if L (+ (equal? item (first L)) (count item (rest L))) 0)))"))

In [175]:
eval(parse("(count 0 (list 0 1 2 3 0 0))"))

3